<a href="https://colab.research.google.com/github/ANEESAAYAZ/Internship_RidgeandLassoRegression/blob/main/Internship_RidgeandLassoRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
pd.set_option("display.width",200)

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv( "properties_features.csv", parse_dates=["listed_date"])
model_df = df[df["price_aed"].notna()].copy()

FORBIDDEN = [ "price_per_sqft", "rental_yield_pct",
              "log_price_aed", "area_price_encoded", "days_on_market",
              "sold_fast", "listing_status", "annual_rent_aed",
              "log_annual_rent_aed"]
DROP = FORBIDDEN + [ "listed_date", "description", "price_aed",
                    "latitude", "longitude"]

X = model_df.drop( columns=[c for c in DROP if c in model_df.columns])
y = np.log1p(model_df["price_aed"])

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

def make_preprocessor():
  return ColumnTransformer([
      ("num", Pipeline([ ("i",SimpleImputer(strategy="median")),
                          ("s", StandardScaler())]), numeric_cols),
      ("cat", Pipeline([("i",SimpleImputer(strategy="most_frequent")),
                        ("e", OneHotEncoder( handle_unknown="ignore",
                                            sparse_output=False,
                                             min_frequency=10 )) ]), categorical_cols),
  ])
X_train, X_test, y_train, y_test = train_test_split( X, y,test_size=0.2, random_state=42)

n_feat = make_preprocessor().fit(X_train).transform(X_train).shape[1]

print(f"\ntrain {len(X_train)} rows, {n_feat} features after encoding")
print(f"ratio: {n_feat / len(X_train):.3f} features per row")


train 2400 rows, 188 features after encoding
ratio: 0.078 features per row


**Ridge Regression**

In [3]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

unscaled = ColumnTransformer([
          ("num", SimpleImputer(strategy="median"), numeric_cols),
          ("cat", Pipeline([ ("i", SimpleImputer(strategy="most_frequent")),
                              ("e", OneHotEncoder(handle_unknown="ignore",
                                                  sparse_output=False,
                                                  min_frequency=10  )) ]), categorical_cols),])

print("\n--- the same Ridge, with and without scaling ---")
print(f"  {'alpha':>8s} {'scaled R2':>11s} {'unscaled R2':>12s}")

for a in [1, 10, 100, 1000, 10000]:
  s = Pipeline([ ("p", make_preprocessor()), ("m",Ridge(alpha=a)) ]).fit(X_train, y_train)
  u = Pipeline([ ("p", unscaled), ("m", Ridge(alpha=a))]).fit(X_train, y_train)
  print(f" {a:8g} {r2_score(y_test, s.predict(X_test)):11.4f} "
        f"{r2_score(y_test, u.predict(X_test)):12.4f}")

scaled_pipe = Pipeline([ ("p", make_preprocessor()), ("m",
                        Ridge(alpha=10.0))]).fit(X_train, y_train)
unscaled_pipe = Pipeline([ ("p", unscaled), ("m",
                          Ridge(alpha=10.0))]).fit(X_train, y_train)

sc = pd.Series( scaled_pipe.named_steps["m"].coef_,index=scaled_pipe.named_steps["p"].get_feature_names_out())

un = pd.Series( unscaled_pipe.named_steps["m"].coef_,index=unscaled_pipe.named_steps["p"].get_feature_names_out())

print("\n  at alpha=10, coefficient on built_up_area_sqft (runs to 17,000):")
print(f"  scaled  {sc['num__built_up_area_sqft']:+.6f}")
print(f"  unscaled {un['num__built_up_area_sqft']:+.6f}")
print("  and on maids_room (a 0/1 flag):")
print(f"  scaled  {sc['num__maids_room']:+.6f}")
print(f"  unscaled {un['num__maids_room']:+.6f}")


--- the same Ridge, with and without scaling ---
     alpha   scaled R2  unscaled R2
        1      0.9446       0.9442
       10      0.9334       0.9336
      100      0.8953       0.8957
     1000      0.8193       0.8246
    10000      0.6051       0.7005

  at alpha=10, coefficient on built_up_area_sqft (runs to 17,000):
  scaled  +0.089733
  unscaled +0.000054
  and on maids_room (a 0/1 flag):
  scaled  +0.055568
  unscaled +0.125994
